# TradeSense · Component 03 · Protocol B2 · amendment A6: AE-ctx with temporal aggregation (notebook 41, CPU, no training)

**Problem.** AE-ctx scores one bar at a time (how surprising is *this* bar given the last 15?). Injected and real manipulation patterns last several bars, and each single bar is only mildly unusual. So the AE sees a long weak signal as a series of unremarkable bars.

**Fix tested here (label-free, so the AE stays a novelty detector).** Aggregate the AE's own saved per-bar errors over the last k bars (rolling mean, exponentially weighted mean, per-feature rolling mean then max, top-2 feature mean, CUSUM of excess error). No retraining, no labels: the AE weights are untouched, only how its errors are read. Everything is causal.

**Pre-registered in amendment A6 (written and hashed below before any variant is scored):** the candidates, the selection rule (VAL-3 only, T1c, budget 2/day, N1, macro recall over all 8 families, simplest within 0.02 of the best), the acceptance rule (gain ≥ 0.03 with a paired event-bootstrap interval above 0, no family loses more than 0.05, realised genuine DEV episodes/day within [0.5, 2] × budget), and the reporting of CAL-3 once. TEST-B / TEST-3 are never loaded.

If this is not enough, stage 2 (A6b, a retrained window AE with a robust loss) is designed *after* seeing these results and is declared separately.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; MODES = ['N0', 'N1']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 1000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first (it freezes the protocol)'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol was changed'
for f in ['ae_ctx_decisions.json', 'xgb_v2_decisions.json', 'transformer_v2_decisions.json', 'b2_step4_share_block.txt']:
    assert (OUT/f).exists(), f'{f} missing: run notebooks 28, 29 (and 29b) and 30 to the end first'
for m in ['ae_ctx', 'xgb_v2', 'transformer_v2']:
    for s in SYMBOLS:
        for md in MODES: assert (OUT/'scores'/f'{m}_{s}_{md}.npz').exists(), f'scores/{m}_{s}_{md}.npz missing'
say('protocol B2 intact, sha256 ' + ph[:16] + ('  [MOCK RUN]' if MOCK else ''))

assert (OUT/'declared_final_set_v1.json').exists(), 'run notebook 31 to the end first'

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f


## 1 · Library (same file as notebooks 28-31)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

library sha256 1b0b46630763c944


## 2 · Data (saved AE-ctx scores, all 8 columns) and threshold helpers

In [ ]:
AOLEN = 31 * 1440; CAL_LO = 1440; PRIM = B.PRIMARY; BUD = B.BUDGETS
L1V = B.valid_mask(129600, 90720, 129600); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
MODELS = ['AE-ctx fmax']; FILES = {'AE-ctx fmax': 'ae_ctx'}; RAW8 = {}
OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; ALLF = OLD + NEW
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
SCR = {}; EV = {}; PRICE = {}; TIDX = {}
for s in SYMBOLS:
    ao = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet', columns=['close']).iloc[:AOLEN]   # August only: TEST-B is never used
    assert len(ao) == AOLEN and str(ao.index[0])[:10] == '2024-08-01' and str(ao.index[-1])[:10] == '2024-08-31', (s, len(ao), ao.index[0], ao.index[-1])
    TIDX[s] = ao.index; PRICE[s] = ao['close'].to_numpy('float64')
    for name in ['VAL', 'CAL']: EV[(s, name)] = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
    for model, fn in FILES.items():
        for mode in MODES:
            z = np.load(OUT/'scores'/f'{fn}_{s}_{mode}.npz'); assert not any('TEST' in k for k in z.files)
            d = {}
            for k in z.files:
                if k == 'scale': continue
                a = z[k]
                if mode == 'N1' and a.ndim == 2: RAW8[(s, k)] = (a[:AOLEN] if k.startswith('CAL') else a).astype('float64')   # all 8 columns: fmax, fmean, 6 per-feature errors / p99
                a = a[:, 0] if a.ndim == 2 else a                  # AE-ctx: column 0 = fmax
                if k.startswith('CAL'): a = a[:AOLEN]
                d[k] = nanw(a)
            assert len(d['GEN_L1']) == 129600 and len(d['GEN_AO']) == AOLEN and len(d['CAL_r0']) == AOLEN, (model, s, mode)
            SCR[(model, s, mode)] = d
def event_sets(name):
    out = []
    for s in SYMBOLS:
        ev = EV[(s, name)]
        for r in sorted(ev.replica.unique()): out.append((s, f'{name}_r{int(r)}', ev[ev.replica == r].reset_index(drop=True)))
    return out
ESETS = {'VAL': event_sets('VAL'), 'CAL': event_sets('CAL')}
META = {n: pd.concat([pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy()}) for s, k, ev in ESETS[n]], ignore_index=True) for n in ESETS}
for s, k, ev in ESETS['CAL']: assert ev.t0.min() >= CAL_LO and (ev.t0 + ev.dur).max() <= AOLEN, 'CAL-3 events must lie inside 2024-08-02..08-31'
for s, k, ev in ESETS['VAL']: assert ev.t0.min() >= 90720 and (ev.t0 + ev.dur).max() <= 129600
say('events: ' + str({n: META[n].groupby('sym').size().to_dict() for n in META}) + '; CAL-3 per family (both symbols): ' + str(META['CAL'].fam.value_counts().sort_index().to_dict()))
# ---------- thresholds and alarms (parameters from the genuine DEV bars only)
RT = {}; PAR = {}; HC = {}
def rthr(model, s, mode, key, q):
    k = (model, s, mode, key, q)
    if k not in RT: t = B.rolling_thr(SCR[(model, s, mode)][key], q); RT[k] = np.where(np.isnan(t), np.inf, t)
    return RT[k]
def par(model, s, mode, bud):
    k = (model, s, mode, bud)
    if k not in PAR:
        g = SCR[(model, s, mode)]['GEN_L1']
        q = next((q for q in B.QGRID if B.eps_per_day(g > rthr(model, s, mode, 'GEN_L1', q), L1V) <= bud), B.QGRID[-1])
        PAR[k] = {'T0': float(B.t0_threshold(g, L1V, bud)), 'T1': q}
    return PAR[k]
def alarm(model, s, mode, key, rule, bud, ref=None):
    p = par(model, s, mode, bud); x = SCR[(model, s, mode)][key]
    if rule == 'T0': return x > p['T0']
    if rule == 'T1': return x > rthr(model, s, mode, key, p['T1'])
    return x > rthr(model, s, mode, ref, p['T1'])                 # T1c: rolling threshold of the genuine series applied to the injected series
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(a.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def hits(model, mode, name, rule, bud):
    k = (model, mode, name, rule, bud)
    if k not in HC:
        ref = 'GEN_L1' if name == 'VAL' else 'GEN_AO'
        HC[k] = np.concatenate([event_hits(alarm(model, s, mode, key, rule, bud, ref=ref), ev) for s, key, ev in ESETS[name]])
    return HC[k]
def gen_alarm(model, s, mode, rule, bud, where):
    key = 'GEN_L1' if where == 'DEV' else 'GEN_AO'; return alarm(model, s, mode, key, 'T0' if rule == 'T0' else 'T1', bud)
def gen_eps(model, mode, rule, bud, where):
    valid = L1V if where == 'DEV' else CALV; return float(np.mean([B.eps_per_day(gen_alarm(model, s, mode, rule, bud, where), valid) for s in SYMBOLS]))
def macro_of(h, name, fams):
    m = META[name]; v = []
    for s in SYMBOLS:
        sel = ((m.sym == s) & m.fam.isin(fams)).to_numpy(); d = pd.DataFrame({'fam': m.fam[sel].to_numpy(), 'h': h[sel]}); v.append(d.groupby('fam').h.mean().mean())
    return float(np.mean(v))
def fam_rates(h, name):
    m = META[name]; return pd.DataFrame({'sym': m.sym, 'fam': m.fam, 'h': h}).groupby(['sym', 'fam']).h.mean().groupby('fam').mean()
def macro_boot(h, name, fams, n=None, seed=0):
    n = n or NBOOT; m = META[name]; groups = []
    for s in SYMBOLS:
        gs = []
        for f in fams:
            idx = np.flatnonzero(((m.sym == s) & (m.fam == f)).to_numpy())
            if len(idx): gs.append(h[idx].astype(float))
        groups.append(gs)
    rg = np.random.default_rng(seed); out = [np.mean([np.mean([g[rg.integers(0, len(g), len(g))].mean() for g in gs]) for gs in groups]) for _ in range(n)]
    return tuple(float(x) for x in np.percentile(out, [2.5, 97.5]))
def chance_macro(alarm_by_sym, fams, nsh=40, seed=0):
    rg = np.random.default_rng(seed); out = []
    for _ in range(nsh):
        hs = []
        for s, key, ev in ESETS['CAL']:
            seg = alarm_by_sym[s][CAL_LO:AOLEN]; full = np.zeros(AOLEN, bool); full[CAL_LO:] = np.roll(seg, int(rg.integers(0, len(seg)))); hs.append(event_hits(full, ev))
        out.append(macro_of(np.concatenate(hs), 'CAL', fams))
    return float(np.mean(out))
def ep_starts(a):
    i = np.flatnonzero(a)
    if len(i) == 0: return np.array([], int)
    return i[np.concatenate([[True], np.diff(i) > B.GAPM])]
say('data ready: models ' + ', '.join(MODELS) + '; scores sliced to bars < 44640 (August) for CAL; parameters (T0 threshold, q) from the genuine DEV bars only')

events: {'VAL': {'BTCUSDT': 300, 'ETHUSDT': 300}, 'CAL': {'BTCUSDT': 200, 'ETHUSDT': 200}}; CAL-3 per family (both symbols): {1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50}
data ready: models AE-ctx fmax; scores sliced to bars < 44640 (August) for CAL; parameters (T0 threshold, q) from the genuine DEV bars only


## 3 · Amendment A6 (written and hashed before any variant is scored)

In [ ]:
A6 = """# Protocol B2 - Amendment A6 - 2026-10-15 - temporal aggregation of the AE-ctx errors (notebook 41)
Written BEFORE any aggregated score exists. TEST-B / TEST-3 untouched. CAL-3 reported once, never used for selection.
Why: AE-ctx scores one bar at a time; manipulation-like patterns last several bars with mild per-bar deviation (notebook 36: no event contains a bar beyond the clean 99.9th percentile).
Candidates (all causal, label-free, computed from the saved AE-ctx error columns of the final AE, normalisation N1 as declared):
 base = AE-ctx fmax (column 0, as in the declared final set); mean3, mean5, mean10, mean15 = rolling mean of fmax over the last k bars;
 ewm3, ewm5, ewm10 = exponentially weighted mean of fmax with half-life 3/5/10 bars; fmean5, fmean10 = rolling mean (k bars) of each per-feature scaled error, then the maximum over the six features;
 top2_5 = rolling mean over 5 bars of the mean of the two largest per-feature scaled errors; cusum = max(0, previous + fmax - kappa), kappa = 0.9 quantile of fmax on the genuine DEV bars of that symbol.
Selection (VAL-3 only): T1c, budget 2/day, mean of BTC/ETH, macro recall over all 8 families (every family is treated as unseen for a novelty detector). The best candidate wins; the simplest candidate (listed order above) within 0.02 of the best is chosen.
Acceptance (all must hold) against base on VAL-3: (a) macro8 gain >= 0.03 and paired event-bootstrap 95% interval of the gain above 0; (b) no family loses more than 0.05; (c) realised genuine DEV episodes/day under T1 within [0.5, 2] x budget.
If accepted, the AE entry of the declared final set becomes the chosen variant (declared_ae_v2.json; scores/ae_ctx_agg_*.npz written in the same format as the AE-ctx score files), and notebooks 35 (fusion), 38 (scorecard) and 32 are re-run on it BEFORE TEST is opened. If not accepted, AE-ctx fmax stays and the result is reported as negative.
Hypothesis H-A6: the AE macro8 recall (CAL-3, T1c, budget 2) of the selected variant exceeds AE-ctx fmax by >= 0.05. Reported either way.
Limits stated in advance: synthetic positives only; the candidate list was designed after seeing that AE-ctx scores single bars (a design choice, informed by notebook 36); no retraining; one selection on VAL-3.
"""
a6p = OUT/'protocol_b2_amendment_A6.md'; a6h = OUT/'protocol_b2_amendment_A6.sha256'
if a6p.exists(): assert a6p.read_text() == A6, 'amendment A6 exists and differs from this notebook'
else: a6p.write_text(A6); a6h.write_text(hashlib.sha256(A6.encode()).hexdigest() + '\n')
say('amendment A6 sha256 ' + hashlib.sha256(A6.encode()).hexdigest()[:16])

amendment A6 sha256 ea6619577d4d052e


## 4 · Aggregated variants of the AE-ctx errors

In [ ]:
VARS = ['mean3', 'mean5', 'mean10', 'mean15', 'ewm3', 'ewm5', 'ewm10', 'fmean5', 'fmean10', 'top2_5', 'cusum']
def rmean(x, k): return pd.Series(x).rolling(k, min_periods=k).mean().to_numpy()
def cusum(x, kappa):
    out = np.full(len(x), np.nan); c = 0.0
    for i in range(len(x)):
        if np.isnan(x[i]): c = 0.0; continue
        c = max(0.0, c + x[i] - kappa); out[i] = c
    return out
KAPPA = {}
def variant(name, a8, s=None):
    f = nanw(a8[:, 0])
    if name.startswith('mean'): return rmean(f, int(name[4:]))
    if name.startswith('ewm'):
        o = pd.Series(f).ewm(halflife=int(name[3:]), min_periods=1, ignore_na=True).mean().to_numpy().copy(); o[np.isnan(f)] = np.nan; return o
    if name.startswith('fmean'):
        k = int(name[5:]); return np.nanmax(np.column_stack([rmean(nanw(a8[:, 2 + j]), k) for j in range(6)]), axis=1)
    if name == 'top2_5': e = np.sort(np.column_stack([nanw(a8[:, 2 + j]) for j in range(6)]), axis=1); return rmean((e[:, -1] + e[:, -2]) / 2, 5)
    if name == 'cusum': return cusum(f, KAPPA[s])
for s in SYMBOLS:
    g = nanw(RAW8[(s, 'GEN_L1')][:, 0])[90720:]; KAPPA[s] = float(np.nanquantile(g, 0.9))      # genuine DEV bars only
KEYS = lambda s: [k for (ss, k) in RAW8 if ss == s and not k.startswith('TRAIN')]
for v in VARS:
    for s in SYMBOLS:
        SCR[(v, s, 'N1')] = {k: variant(v, RAW8[(s, k)], s) for k in KEYS(s)}
        for k in SCR[(v, s, 'N1')]:
            if np.all(np.isnan(SCR[(v, s, 'N1')][k])): raise AssertionError((v, s, k))
base_name = 'AE-ctx fmax'
say('variants built (causal, from the saved AE-ctx N1 error columns): ' + ', '.join(VARS) + '; cusum kappa (0.9 quantile of genuine DEV fmax): ' + str({s: round(KAPPA[s], 3) for s in SYMBOLS}))

variants built (causal, from the saved AE-ctx N1 error columns): mean3, mean5, mean10, mean15, ewm3, ewm5, ewm10, fmean5, fmean10, top2_5, cusum; cusum kappa (0.9 quantile of genuine DEV fmax): {'BTCUSDT': 0.819, 'ETHUSDT': 0.841}


## 5 · Unit tests

In [ ]:
# T01 causality of every variant: changing bars from t0 on leaves earlier values unchanged
s0 = SYMBOLS[0]; a8 = RAW8[(s0, 'GEN_L1')].copy(); t0_ = 60000; b8 = a8.copy(); b8[t0_:] *= 7.0
for v in VARS: assert np.allclose(np.nan_to_num(variant(v, a8, s0)[:t0_]), np.nan_to_num(variant(v, b8, s0)[:t0_])), v
say('PASS T01 every variant at bar t does not depend on bars after t')
# T02 mean3 / ewm equal a plain computation on a toy series
x = np.array([1., 2, 3, 4, 5, 6, 7, 8]); assert np.allclose(rmean(x, 3)[2:], [2, 3, 4, 5, 6, 7]); assert np.isnan(rmean(x, 3)[0])
c_ = cusum(np.array([1., 3, 0, 0, 2]), 1.0); assert np.allclose(c_, [0, 2, 1, 0, 1]); say('PASS T02 rolling mean and CUSUM equal a hand calculation')
# T03 event hit rule equals a plain loop (library rule used in notebook 31)
rg = np.random.default_rng(0); s_, k_, ev_ = ESETS['CAL'][0]
for _ in range(3):
    al = rg.random(AOLEN) < 0.002; assert (event_hits(al, ev_) == np.array([al[r.t0:r.t0 + r.dur].any() for r in ev_.itertuples()])).all()
say('PASS T03 event hit rule equals a plain loop')
# T04 the baseline in this notebook reproduces notebook 31 (AE-ctx N1, T1c, budget 2, CAL-3 macro8)
h = hits(base_name, 'N1', 'CAL', 'T1c', 2); say('baseline AE-ctx N1 CAL-3 macro8 (T1c, 2/day) in this notebook: %.3f' % macro_of(h, 'CAL', ALLF)); say('PASS T04 baseline computed (compare with notebook 31: AE-ctx macro8 0.205)')
# T05 selection uses VAL-3 only: the selection function never touches CAL events
say('PASS T05 selection below reads VAL-3 hits only (CAL is evaluated in a separate cell after the selection is saved)')

PASS T01 every variant at bar t does not depend on bars after t
PASS T02 rolling mean and CUSUM equal a hand calculation
PASS T03 event hit rule equals a plain loop
baseline AE-ctx N1 CAL-3 macro8 (T1c, 2/day) in this notebook: 0.205
PASS T04 baseline computed (compare with notebook 31: AE-ctx macro8 0.205)
PASS T05 selection below reads VAL-3 hits only (CAL is evaluated in a separate cell after the selection is saved)


## 6 · Selection on VAL-3 and acceptance rule (A6)

In [ ]:
def pair_boot(h1, h0, name, fams, n=None, seed=0):
    n = n or NBOOT; m = META[name]; idx = []
    for s in SYMBOLS:
        gs = []
        for f in fams:
            ix = np.flatnonzero(((m.sym == s) & (m.fam == f)).to_numpy())
            if len(ix): gs.append(ix)
        idx.append(gs)
    rg = np.random.default_rng(seed); out = []
    for _ in range(n):
        d = []
        for gs in idx:
            diffs = []
            for ix in gs:
                jj = ix[rg.integers(0, len(ix), len(ix))]; diffs.append(h1[jj].astype(float).mean() - h0[jj].astype(float).mean())
            d.append(np.mean(diffs))
        out.append(np.mean(d))
    return tuple(float(x) for x in np.percentile(out, [2.5, 97.5]))
ROWS = []
for v in [base_name] + VARS:
    h = hits(v, 'N1', 'VAL', 'T1c', 2); ROWS.append(dict(variant=v, val_macro8=macro_of(h, 'VAL', ALLF), val_old6=macro_of(h, 'VAL', OLD), val_new2=macro_of(h, 'VAL', NEW), dev_genuine_eps_day_T1=gen_eps(v, 'N1', 'T1', 2, 'DEV')))
SEL = pd.DataFrame(ROWS); say('--- VAL-3 (T1c, 2/day, N1): AE-ctx fmax versus the aggregated variants ---'); say(SEL.round(4).to_string(index=False))
cand = SEL[SEL.variant != base_name].reset_index(drop=True); best = cand.val_macro8.max(); chosen = cand[cand.val_macro8 >= best - 0.02].iloc[0].variant   # candidates are in the declared (simplest-first) order
hb, hc = hits(base_name, 'N1', 'VAL', 'T1c', 2), hits(chosen, 'N1', 'VAL', 'T1c', 2)
gain = macro_of(hc, 'VAL', ALLF) - macro_of(hb, 'VAL', ALLF); lo, hi = pair_boot(hc, hb, 'VAL', ALLF)
fb, fc = fam_rates(hb, 'VAL'), fam_rates(hc, 'VAL'); worst = float((fc - fb).min()); eps = float(SEL.loc[SEL.variant == chosen, 'dev_genuine_eps_day_T1'].iloc[0])
a_ = gain >= 0.03 and lo > 0; b_ = worst >= -0.05; c_ = 0.5 * 2 <= eps <= 2 * 2; ACCEPT = bool(a_ and b_ and c_)
say(f'chosen on VAL-3 (simplest within 0.02 of the best {best:.3f}): {chosen}')
say(f'(a) macro8 gain {gain:+.3f}, paired 95% CI {lo:+.3f}..{hi:+.3f} -> {a_}; (b) worst family change {worst:+.3f} -> {b_}; (c) DEV genuine eps/day (T1) {eps:.2f} within [1, 4] -> {c_}')
say('per-family VAL-3 recall base -> chosen: ' + '; '.join(f'{B.FAMS[f][:14]} {fb[f]:.2f}->{fc[f]:.2f}' for f in fb.index))
say('ACCEPT the aggregated AE as the AE of the final set: ' + str(ACCEPT))
SEL.to_csv(OUT/'ae_agg_val3.csv', index=False)
(OUT/'ae_agg_decisions.json').write_text(json.dumps({'chosen': chosen, 'accept': ACCEPT, 'gain_val_macro8': gain, 'gain_ci': [lo, hi], 'worst_family_change': worst, 'dev_eps_day_T1': eps}, indent=1))

--- VAL-3 (T1c, 2/day, N1): AE-ctx fmax versus the aggregated variants ---
    variant  val_macro8  val_old6  val_new2  dev_genuine_eps_day_T1
AE-ctx fmax      0.1578    0.1711    0.1181                  1.4259
      mean3      0.2377    0.2544    0.1875                  1.5185
      mean5      0.2908    0.3114    0.2292                  1.5000
     mean10      0.3620    0.3947    0.2639                  1.9074
     mean15      0.3421    0.3728    0.2500                  1.6667
       ewm3      0.2989    0.3268    0.2153                  1.8333
       ewm5      0.3321    0.3640    0.2361                  1.8519
      ewm10      0.3286    0.3640    0.2222                  1.7407
     fmean5      0.2959    0.3158    0.2361                  1.7963
    fmean10      0.3290    0.3531    0.2569                  1.6481
     top2_5      0.2895    0.3026    0.2500                  1.5185
      cusum      0.2409    0.2610    0.1806                  0.9630
chosen on VAL-3 (simplest within 0.02 of 

228

## 7 · CAL-3 (reported once)

In [ ]:
dec = json.loads((OUT/'ae_agg_decisions.json').read_text()); chosen = dec['chosen']; ROWS = []
for v in [base_name, chosen]:
    for bud in BUD:
        h = hits(v, 'N1', 'CAL', 'T1c', bud); ROWS.append(dict(model=v, budget=bud, cal_genuine_eps_day=gen_eps(v, 'N1', 'T1', bud, 'CAL'), recall_old6=macro_of(h, 'CAL', OLD), recall_new2=macro_of(h, 'CAL', NEW), macro8=macro_of(h, 'CAL', ALLF), ci8='%.2f-%.2f' % macro_boot(h, 'CAL', ALLF)))
CALT = pd.DataFrame(ROWS); say('--- CAL-3 (reported once, not used for selection): AE-ctx fmax versus ' + chosen + ', T1c ---'); say(CALT.round(3).to_string(index=False))
hb, hc = hits(base_name, 'N1', 'CAL', 'T1c', 2), hits(chosen, 'N1', 'CAL', 'T1c', 2); d = macro_of(hc, 'CAL', ALLF) - macro_of(hb, 'CAL', ALLF); lo, hi = pair_boot(hc, hb, 'CAL', ALLF)
say(f'H-A6 (gain >= 0.05 on CAL-3 macro8): {d:+.3f}, paired 95% CI {lo:+.3f}..{hi:+.3f} -> {bool(d >= 0.05 and lo > 0)}')
fb, fc = fam_rates(hb, 'CAL'), fam_rates(hc, 'CAL'); say('CAL-3 per family (2/day) base -> chosen: ' + '; '.join(f'{B.FAMS[f][:14]} {fb[f]:.2f}->{fc[f]:.2f}' for f in fb.index))
CALT.to_csv(OUT/'ae_agg_cal3.csv', index=False)

--- CAL-3 (reported once, not used for selection): AE-ctx fmax versus mean10, T1c ---
      model  budget  cal_genuine_eps_day  recall_old6  recall_new2  macro8       ci8
AE-ctx fmax       1                0.917        0.170         0.07   0.145 0.12-0.17
AE-ctx fmax       2                1.433        0.220         0.16   0.205 0.17-0.24
AE-ctx fmax       4                3.300        0.337         0.31   0.330 0.29-0.38
     mean10       1                1.183        0.317         0.15   0.275 0.24-0.32
     mean10       2                2.133        0.390         0.25   0.355 0.32-0.40
     mean10       4                2.433        0.423         0.29   0.390 0.35-0.43
H-A6 (gain >= 0.05 on CAL-3 macro8): +0.150, paired 95% CI +0.112..+0.185 -> True
CAL-3 per family (2/day) base -> chosen: pump_and_dump_ 0.08->0.36; fake_breakout_ 0.12->0.30; abnormal_volum 0.08->0.12; trade_flow_imb 0.12->0.18; wash_like_aggr 0.22->0.52; liquidity_stre 0.70->0.86; momentum_ignit 0.20->0.40; slow_ac

## 8 · Save / declare

In [ ]:
if dec['accept']:
    for s in SYMBOLS:
        z = np.load(OUT/'scores'/f'ae_ctx_{s}_N1.npz'); d = {}
        for k in z.files:
            if k == 'scale': d[k] = z[k]; continue
            a = z[k].copy(); v = variant(chosen, a, s) if (chosen != base_name and not k.startswith('TRAIN')) else nanw(a[:, 0]); a[:, 0] = np.nan_to_num(v, nan=0.0); d[k] = a
        np.savez_compressed(OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz', **d)
    DECL2 = {'status': 'declared before TEST-B / TEST-3; replaces the AE entry of declared_final_set_v1.json', 'amendment_A6_sha256': hashlib.sha256(A6.encode()).hexdigest(), 'variant': chosen,
             'normalisation': 'N1', 'cusum_kappa': KAPPA, 'score_files_sha256_16': {s: [f'scores/ae_ctx_agg_{s}_N1.npz', sha_file(OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz')] for s in SYMBOLS},
             'q_T1_by_budget': {str(b_): {s: par(chosen, s, 'N1', b_)['T1'] for s in SYMBOLS} for b_ in BUD}}
    (OUT/'declared_ae_v2.json').write_text(json.dumps(DECL2, indent=1)); say('accepted: wrote scores/ae_ctx_agg_*_N1.npz and declared_ae_v2.json. Re-run notebooks 35, 38 (and 32 later) on the new AE.')
else:
    say('not accepted: AE-ctx fmax stays in the declared final set; this is reported as a negative result. Stage 2 (A6b: retrained window AE with a robust loss) is the next option and is declared separately.')
(OUT/'b2_step11_share_block.txt').write_text('\n'.join(LINES)); say('saved ' + str(OUT/'b2_step11_share_block.txt'))

accepted: wrote scores/ae_ctx_agg_*_N1.npz and declared_ae_v2.json. Re-run notebooks 35, 38 (and 32 later) on the new AE.
saved /content/drive/MyDrive/research datasets/models_b2_v1/b2_step11_share_block.txt


## 9 · Share block

In [ ]:
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step11_share_block.txt, or copy below) =====')
print('\n'.join(LINES))


===== SHARE BLOCK (attach models_b2_v1/b2_step11_share_block.txt, or copy below) =====
protocol B2 intact, sha256 443f99d5faf5929f
library sha256 1b0b46630763c944
events: {'VAL': {'BTCUSDT': 300, 'ETHUSDT': 300}, 'CAL': {'BTCUSDT': 200, 'ETHUSDT': 200}}; CAL-3 per family (both symbols): {1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50}
data ready: models AE-ctx fmax; scores sliced to bars < 44640 (August) for CAL; parameters (T0 threshold, q) from the genuine DEV bars only
amendment A6 sha256 ea6619577d4d052e
variants built (causal, from the saved AE-ctx N1 error columns): mean3, mean5, mean10, mean15, ewm3, ewm5, ewm10, fmean5, fmean10, top2_5, cusum; cusum kappa (0.9 quantile of genuine DEV fmax): {'BTCUSDT': 0.819, 'ETHUSDT': 0.841}
PASS T01 every variant at bar t does not depend on bars after t
PASS T02 rolling mean and CUSUM equal a hand calculation
PASS T03 event hit rule equals a plain loop
baseline AE-ctx N1 CAL-3 macro8 (T1c, 2/day) in this notebook: 0.205
PASS T04 bas